# rnn_walk_forward_validation_tuning

Recurrent neural network validation tuning using the full-history session-aligned dataset.

This notebook uses compact GRU/LSTM sequence models. Instead of giving the model a single engineered row, it builds rolling ticker-specific sequences ending on the prediction date. The default grid is intentionally smaller than the SVM/RF/logistic grids because each candidate requires neural-network training.

The pipeline keeps the same evaluation discipline as the other notebooks: it splits on the full session calendar before removing neutral targets, selects feature/hyperparameter configurations with expanding-window walk-forward validation, calibrates the final probability threshold on the holdout validation period, and evaluates the frozen model on the untouched test split.


In [1]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers
except ImportError as exc:
    tf = None
    keras = None
    layers = None
    regularizers = None
    TENSORFLOW_IMPORT_ERROR = exc
else:
    TENSORFLOW_IMPORT_ERROR = None

if tf is None:
    raise ImportError(
        "TensorFlow is required for this RNN notebook. Update the project environment from environment.yml "
        "or install into the active notebook kernel with: python -m pip install \"tensorflow>=2.16,<2.18\""
    ) from TENSORFLOW_IMPORT_ERROR

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 280)


In [2]:
from __future__ import annotations

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_utils.experiment_config import build_default_config
from notebook_utils.feature_set_grid_builder import FeatureFrameBuilder, FeatureSetGridBuilder
from notebook_utils.metrics import ClassificationMetrics
from notebook_utils.model_report_builder import ModelReportBuilder
from notebook_utils.split_utils import make_split_dates, make_walk_forward_fold_specs, subset_by_dates

CONFIG = build_default_config(PROJECT_ROOT)

RNN_PARAM_GRID = [
    {
        "param_set": "gru_len5_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced",
        "sequence_length": 5,
        "cell_type": "GRU",
        "hidden_units": 8,
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 1e-3,
        "batch_size": 128,
        "max_epochs": 80,
        "patience": 8,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "gru_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced",
        "sequence_length": 10,
        "cell_type": "GRU",
        "hidden_units": 8,
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 1e-3,
        "batch_size": 128,
        "max_epochs": 80,
        "patience": 8,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "gru_len20_h16_drop0p3_l2_1e-3_lr5e-4_b128_balanced",
        "sequence_length": 20,
        "cell_type": "GRU",
        "hidden_units": 16,
        "dropout": 0.3,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 5e-4,
        "batch_size": 128,
        "max_epochs": 100,
        "patience": 10,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "lstm_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced",
        "sequence_length": 10,
        "cell_type": "LSTM",
        "hidden_units": 8,
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "dense_units": 0,
        "dense_dropout": 0.0,
        "l2": 1e-3,
        "learning_rate": 1e-3,
        "batch_size": 128,
        "max_epochs": 80,
        "patience": 8,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b128_balanced",
        "sequence_length": 10,
        "cell_type": "GRU",
        "hidden_units": 16,
        "dropout": 0.3,
        "recurrent_dropout": 0.0,
        "dense_units": 8,
        "dense_dropout": 0.2,
        "l2": 1e-2,
        "learning_rate": 5e-4,
        "batch_size": 128,
        "max_epochs": 100,
        "patience": 10,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "lstm_len20_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b128_balanced",
        "sequence_length": 20,
        "cell_type": "LSTM",
        "hidden_units": 16,
        "dropout": 0.3,
        "recurrent_dropout": 0.0,
        "dense_units": 8,
        "dense_dropout": 0.2,
        "l2": 1e-2,
        "learning_rate": 5e-4,
        "batch_size": 128,
        "max_epochs": 100,
        "patience": 10,
        "class_weight_mode": "balanced",
    },
]

pd.DataFrame(RNN_PARAM_GRID)

feature_grid = FeatureSetGridBuilder.build(
    max_features_per_model=CONFIG["max_features_per_model"],
)

PRICE_FEATURES = feature_grid.price_features
VOLUME_FEATURE_OPTIONS = feature_grid.volume_feature_options
GDELT_FEATURE_OPTIONS = feature_grid.gdelt_feature_options
GDELT_SENTIMENT_FEATURE_OPTIONS = feature_grid.gdelt_sentiment_feature_options
GDELT_ATTENTION_FEATURE_OPTIONS = feature_grid.gdelt_attention_feature_options
REDDIT_FEATURE_OPTIONS = feature_grid.reddit_feature_options
REDDIT_ATTENTION_FEATURE_OPTIONS = feature_grid.reddit_attention_feature_options
GOOGLE_TRENDS_FEATURE_OPTIONS = feature_grid.google_trends_feature_options
GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS = feature_grid.google_score_attention_feature_options
DERIVED_FEATURE_COLUMNS = feature_grid.derived_feature_columns
BASE_VOLUME_OPTION = feature_grid.base_volume_option
BASELINE_FEATURE_SET = feature_grid.baseline_feature_set
FEATURE_SET_SPECS = feature_grid.feature_set_specs
FEATURE_SETS = feature_grid.feature_sets
FEATURE_SET_METADATA = feature_grid.feature_set_metadata
SKIPPED_FEATURE_SETS = feature_grid.skipped_feature_sets
FEATURE_SETS_TO_TEST = feature_grid.feature_sets_to_test
pd.DataFrame(RNN_PARAM_GRID)


,param_set,sequence_length,cell_type,hidden_units,dropout,recurrent_dropout,dense_units,dense_dropout,l2,learning_rate,batch_size,max_epochs,patience,class_weight_mode
0,gru_len5_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced,5,GRU,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced
1,gru_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced,10,GRU,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced
2,gru_len20_h16_drop0p3_l2_1e-3_lr5e-4_b128_bala...,20,GRU,16,0.3,0.0,0,0.0,0.001,0.0005,128,100,10,balanced
3,lstm_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_bala...,10,LSTM,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced
4,gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b1...,10,GRU,16,0.3,0.0,8,0.2,0.010,0.0005,128,100,10,balanced
5,lstm_len20_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b...,20,LSTM,16,0.3,0.0,8,0.2,0.010,0.0005,128,100,10,balanced


In [3]:
raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = FeatureFrameBuilder.build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)
walk_forward_fold_specs = make_walk_forward_fold_specs(
    train_dates,
    n_folds=CONFIG["walk_forward_folds"],
    validation_size=CONFIG["walk_forward_validation_dates"],
    min_train_dates=CONFIG["walk_forward_min_train_dates"],
    gap_days=CONFIG["gap_days"],
)

split_date_map = {
    "train": train_dates,
    "validation": validation_dates,
    "test": test_dates,
}
feature_df["split"] = "gap"
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df["date"].isin(split_dates), "split"] = split_name

modeled_df = feature_df[feature_df["target"].isin([0.0, 1.0])].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)
train_validation_df = subset_by_dates(modeled_df, list(train_dates) + list(validation_dates))

split_summary_rows = []
for split_name in ["train", "validation", "test"]:
    all_split_df = feature_df[feature_df["split"].eq(split_name)]
    modeled_split_df = modeled_df[modeled_df["split"].eq(split_name)]
    split_summary_rows.append(
        {
            "split": split_name,
            "session_rows": len(all_split_df),
            "modeled_rows": len(modeled_split_df),
            "session_dates": all_split_df["date"].nunique(),
            "modeled_dates": modeled_split_df["date"].nunique(),
            "date_min": all_split_df["date"].min(),
            "date_max": all_split_df["date"].max(),
            "target_positive_rate": modeled_split_df["target"].mean(),
        }
    )

split_summary_df = pd.DataFrame(split_summary_rows)
walk_forward_fold_summary_df = pd.DataFrame(
    [
        {
            "fold": spec["fold"],
            "train_n_dates": spec["train_n_dates"],
            "train_date_min": spec["train_date_min"],
            "train_date_max": spec["train_date_max"],
            "validation_n_dates": spec["validation_n_dates"],
            "validation_date_min": spec["validation_date_min"],
            "validation_date_max": spec["validation_date_max"],
        }
        for spec in walk_forward_fold_specs
    ]
)

split_summary_df


,split,session_rows,modeled_rows,session_dates,modeled_dates,date_min,date_max,target_positive_rate
0,train,5632,4456,704,704,2021-01-04,2023-10-19,0.511670
1,validation,1880,1424,235,235,2023-10-23,2024-09-27,0.568118
2,test,2512,1886,314,313,2024-10-01,2025-12-31,0.526511


In [4]:
walk_forward_fold_summary_df

,fold,train_n_dates,train_date_min,train_date_max,validation_n_dates,validation_date_min,validation_date_max
0,1,383,2021-01-04,2022-07-12,80,2022-07-14,2022-11-03
1,2,463,2021-01-04,2022-11-02,80,2022-11-04,2023-03-02
2,3,543,2021-01-04,2023-03-01,80,2023-03-03,2023-06-27
3,4,623,2021-01-04,2023-06-26,80,2023-06-28,2023-10-19


In [5]:
neutral_summary_by_ticker_df = (
    feature_df.groupby("ticker")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reset_index()
)
neutral_summary_by_ticker_df["neutral_rate_among_available"] = (
    neutral_summary_by_ticker_df["neutral"] / neutral_summary_by_ticker_df["target_available"]
)
neutral_summary_by_ticker_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_ticker_df["neutral_rate_among_available"]

neutral_summary_by_split_df = (
    feature_df[feature_df["split"].isin(["train", "validation", "test"])]
    .groupby("split")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
neutral_summary_by_split_df["neutral_rate_among_available"] = (
    neutral_summary_by_split_df["neutral"] / neutral_summary_by_split_df["target_available"]
)
neutral_summary_by_split_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_split_df["neutral_rate_among_available"]

print("Neutral coverage by split")
print(neutral_summary_by_split_df.to_string(index=False))
print("\nNeutral coverage by ticker")
neutral_summary_by_ticker_df

Neutral coverage by split
     split  rows  target_available  neutral  neutral_rate_among_available  modeled_rate_among_available
     train  5632              5632     1176                      0.208807                      0.791193
validation  1880              1880      456                      0.242553                      0.757447
      test  2512              2504      618                      0.246805                      0.753195

Neutral coverage by ticker


,ticker,rows,target_available,neutral,neutral_rate_among_available,modeled_rate_among_available
0,AAPL,1255,1254,378,0.301435,0.698565
1,AMD,1255,1254,220,0.175439,0.824561
2,AMZN,1255,1254,300,0.239234,0.760766
3,GOOGL,1255,1254,319,0.254386,0.745614
4,META,1255,1254,278,0.221691,0.778309
5,MSFT,1255,1254,400,0.318979,0.681021
6,NVDA,1255,1254,173,0.137959,0.862041
7,TSLA,1255,1254,184,0.146730,0.853270


In [6]:
missing_requested_feature_sets = [name for name in FEATURE_SETS_TO_TEST if name not in FEATURE_SETS]
if missing_requested_feature_sets:
    raise KeyError(f"Unknown feature sets: {missing_requested_feature_sets}")

missing_feature_columns = sorted(
    {
        feature
        for name in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[name]
        if feature not in feature_df.columns and feature not in DERIVED_FEATURE_COLUMNS
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(FEATURE_SETS[feature_set_name]),
            "features": FEATURE_SETS[feature_set_name],
        }
        for feature_set_name in FEATURE_SETS_TO_TEST
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

all_available_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(features),
            "features": features,
        }
        for feature_set_name, features in FEATURE_SETS.items()
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

skipped_feature_sets_df = pd.DataFrame(SKIPPED_FEATURE_SETS)

print(f"Selection metric: {CONFIG['selection_metric']}")
print(f"Primary validation metric: {CONFIG['primary_validation_metric']}")
print(f"Walk-forward folds: {len(walk_forward_fold_specs)}")
print(f"Tune decision threshold in each validation fold: {CONFIG['tune_decision_threshold']}")
print(f"Max features per model: {CONFIG['max_features_per_model']}")
print(f"Available feature sets: {len(FEATURE_SETS)}")
print(f"RNN feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"Attention feature sets to test: {sum('attention' in FEATURE_SET_METADATA[name]['feature_family'] for name in FEATURE_SETS_TO_TEST)}")
print(f"Skipped feature sets above max feature limit: {len(SKIPPED_FEATURE_SETS)}")
print(f"RNN parameter sets: {len(RNN_PARAM_GRID)}")
print(f"Walk-forward validation fits: {len(FEATURE_SETS_TO_TEST) * len(RNN_PARAM_GRID) * len(walk_forward_fold_specs)}")

candidate_feature_sets_df


Selection metric: balanced_accuracy
Primary validation metric: balanced_accuracy
Walk-forward folds: 4
Tune decision threshold in each validation fold: True
Max features per model: 10
Available feature sets: 61
RNN feature sets to test: 61
Attention feature sets to test: 23
Skipped feature sets above max feature limit: 0
RNN parameter sets: 6
Walk-forward validation fits: 1464


,feature_set,feature_family,n_features,features
0,Model B - price + volume | volume log1p zscore...,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
1,Model B - price + volume | volume percentile r...,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
2,Model B - price + volume | volume zscore 10d,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
3,Model B - price + volume | volume zscore 20d,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
4,Model B - price + volume | volume zscore 20d c...,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
...,...,...,...,...
56,Model F - price + volume + all alternative dat...,price + volume + all alternative data,10,"[return_1d, return_5d, return_20d, rolling_vol..."
57,Model N - price + volume + all attention | per...,price + volume + all attention,8,"[return_1d, return_5d, return_20d, rolling_vol..."
58,Model N - price + volume + all attention | zsc...,price + volume + all attention,8,"[return_1d, return_5d, return_20d, rolling_vol..."
59,Model N - price + volume + all attention | zsc...,price + volume + all attention,8,"[return_1d, return_5d, return_20d, rolling_vol..."


In [7]:
from __future__ import annotations


RNN_PARAM_COLUMNS = [
    "sequence_length",
    "cell_type",
    "hidden_units",
    "dropout",
    "recurrent_dropout",
    "dense_units",
    "dense_dropout",
    "l2",
    "learning_rate",
    "batch_size",
    "max_epochs",
    "patience",
    "class_weight_mode",
]


class SequencePreprocessor:
    def fit(self, X: np.ndarray) -> "SequencePreprocessor":
        flat = X.reshape(-1, X.shape[-1]).astype(float)
        self.median_ = np.nanmedian(flat, axis=0)
        self.median_ = np.where(np.isfinite(self.median_), self.median_, 0.0)
        filled = np.where(np.isnan(flat), self.median_, flat)
        self.mean_ = filled.mean(axis=0)
        self.std_ = filled.std(axis=0)
        self.std_ = np.where((self.std_ > 0.0) & np.isfinite(self.std_), self.std_, 1.0)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        filled = np.where(np.isnan(X), self.median_, X)
        scaled = (filled - self.mean_) / self.std_
        return scaled.astype("float32")

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)


def prepare_train_eval_feature_frames(
    features: list[str],
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "google_trends_above_ticker_train_median" in features:
        return FeatureFrameBuilder.add_google_trends_train_median_feature(train_input_df, eval_input_df)
    return train_input_df, eval_input_df


def prepare_sequence_source_feature_frame(
    features: list[str],
    train_input_df: pd.DataFrame,
    sequence_source_df: pd.DataFrame,
) -> pd.DataFrame:
    if "google_trends_above_ticker_train_median" in features:
        _, sequence_source_with_flag_df = FeatureFrameBuilder.add_google_trends_train_median_feature(
            train_input_df,
            sequence_source_df,
        )
        return sequence_source_with_flag_df
    return sequence_source_df


def params_from_result_row(row: dict | pd.Series) -> dict:
    return {
        "param_set": row["param_set"],
        "sequence_length": int(row["sequence_length"]),
        "cell_type": row["cell_type"],
        "hidden_units": int(row["hidden_units"]),
        "dropout": float(row["dropout"]),
        "recurrent_dropout": float(row["recurrent_dropout"]),
        "dense_units": int(row["dense_units"]),
        "dense_dropout": float(row["dense_dropout"]),
        "l2": float(row["l2"]),
        "learning_rate": float(row["learning_rate"]),
        "batch_size": int(row["batch_size"]),
        "max_epochs": int(row["max_epochs"]),
        "patience": int(row["patience"]),
        "class_weight_mode": row.get("class_weight_mode", "balanced"),
    }


def class_weight_from_target(y_train: np.ndarray, mode: str | None) -> dict[int, float] | None:
    if mode in [None, "none"]:
        return None
    counts = pd.Series(y_train).value_counts()
    n_total = float(len(y_train))
    n_negative = float(counts.get(0, 0.0))
    n_positive = float(counts.get(1, 0.0))
    if n_negative <= 0.0 or n_positive <= 0.0:
        return None
    return {0: n_total / (2.0 * n_negative), 1: n_total / (2.0 * n_positive)}


def split_train_for_early_stopping(train_input_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    dates = np.array(sorted(train_input_df["date"].unique()))
    n_dates = len(dates)
    n_validation = max(
        CONFIG["min_early_stopping_dates"],
        int(round(n_dates * CONFIG["early_stopping_fraction"])),
    )
    n_validation = min(n_validation, max(1, n_dates - CONFIG["min_fit_dates"]))
    fit_dates = dates[:-n_validation]
    early_stop_dates = dates[-n_validation:]
    return subset_by_dates(train_input_df, fit_dates), subset_by_dates(train_input_df, early_stop_dates)


def make_sequence_dataset(
    sequence_source_df: pd.DataFrame,
    sample_df: pd.DataFrame,
    features: list[str],
    sequence_length: int,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    source_lookup = {}
    for ticker, ticker_df in sequence_source_df.sort_values(["ticker", "date"]).groupby("ticker", sort=False):
        dates = [pd.Timestamp(value) for value in ticker_df["date"]]
        source_lookup[ticker] = {
            "positions": {date: position for position, date in enumerate(dates)},
            "values": ticker_df[features].to_numpy(dtype=float),
        }

    sequences = []
    targets = []
    metadata_rows = []
    for row in sample_df.sort_values(["date", "ticker"]).itertuples(index=False):
        ticker_data = source_lookup.get(row.ticker)
        if ticker_data is None:
            continue
        position = ticker_data["positions"].get(pd.Timestamp(row.date))
        if position is None:
            continue
        start = position - sequence_length + 1
        if start < 0:
            continue
        sequences.append(ticker_data["values"][start : position + 1])
        targets.append(int(row.target))
        metadata_rows.append({"date": row.date, "ticker": row.ticker, "target": int(row.target)})

    if not sequences:
        raise ValueError(f"No sequences were built for sequence_length={sequence_length}")
    return np.stack(sequences).astype(float), np.asarray(targets, dtype=int), pd.DataFrame(metadata_rows)


def build_rnn_model(params: dict, n_features: int) -> keras.Model:
    if tf is None:
        raise ImportError(
            "TensorFlow is not installed in this Python environment. Install TensorFlow before running the RNN notebook."
        ) from TENSORFLOW_IMPORT_ERROR

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(CONFIG["random_state"])
    regularizer = regularizers.l2(params["l2"]) if params["l2"] > 0 else None
    inputs = keras.Input(shape=(params["sequence_length"], n_features))
    cell_kwargs = {
        "units": params["hidden_units"],
        "dropout": params["dropout"],
        "recurrent_dropout": params["recurrent_dropout"],
        "kernel_regularizer": regularizer,
        "recurrent_regularizer": regularizer,
    }
    if params["cell_type"] == "GRU":
        x = layers.GRU(**cell_kwargs)(inputs)
    elif params["cell_type"] == "LSTM":
        x = layers.LSTM(**cell_kwargs)(inputs)
    else:
        raise ValueError(f"Unsupported recurrent cell type: {params['cell_type']}")

    if params["dense_units"] > 0:
        x = layers.Dense(params["dense_units"], activation="relu", kernel_regularizer=regularizer)(x)
        if params["dense_dropout"] > 0:
            x = layers.Dropout(params["dense_dropout"])(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=params["learning_rate"]),
        loss="binary_crossentropy",
    )
    return model


def best_threshold_for_balanced_accuracy(y_true: pd.Series, scores: np.ndarray) -> tuple[float, float]:
    return ClassificationMetrics.best_threshold_for_balanced_accuracy(
        y_true,
        scores,
        min_quantile=CONFIG["threshold_min_quantile"],
        max_quantile=CONFIG["threshold_max_quantile"],
        grid_size=CONFIG["threshold_grid_size"],
        default_threshold=0.5,
    )


def metrics_from_scores(y_true: pd.Series, scores: np.ndarray, threshold: float) -> dict:
    return ClassificationMetrics.metrics_from_scores(y_true, scores, threshold)


def add_param_columns(row: dict, params: dict, param_columns: list[str]) -> None:
    for column in param_columns:
        row[column] = params.get(column)


def evaluate_rnn_params(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    decision_threshold: float | None = None,
    tune_threshold: bool = False,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    train_features_df, eval_features_df = prepare_train_eval_feature_frames(
        features,
        train_input_df,
        eval_input_df,
    )
    fit_input_df, early_stop_input_df = split_train_for_early_stopping(train_features_df)
    sequence_source_df = prepare_sequence_source_feature_frame(features, train_input_df, feature_df)

    sequence_length = params["sequence_length"]
    X_fit_raw, y_fit, _ = make_sequence_dataset(sequence_source_df, fit_input_df, features, sequence_length)
    X_early_raw, y_early, _ = make_sequence_dataset(sequence_source_df, early_stop_input_df, features, sequence_length)
    X_eval_raw, y_eval, eval_metadata_df = make_sequence_dataset(sequence_source_df, eval_features_df, features, sequence_length)

    preprocessor = SequencePreprocessor()
    X_fit = preprocessor.fit_transform(X_fit_raw)
    X_early = preprocessor.transform(X_early_raw)
    X_eval = preprocessor.transform(X_eval_raw)

    model = build_rnn_model(params, n_features=len(features))
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=params["patience"],
            restore_best_weights=True,
        )
    ]
    history = model.fit(
        X_fit,
        y_fit,
        validation_data=(X_early, y_early),
        epochs=params["max_epochs"],
        batch_size=params["batch_size"],
        verbose=0,
        callbacks=callbacks,
        class_weight=class_weight_from_target(y_fit, params.get("class_weight_mode")),
    )
    scores = model.predict(X_eval, batch_size=params["batch_size"], verbose=0).reshape(-1)

    if tune_threshold:
        decision_threshold, _ = best_threshold_for_balanced_accuracy(
            y_eval,
            scores,
        )
    elif decision_threshold is None:
        decision_threshold = 0.5

    metric_result = metrics_from_scores(y_eval, scores, float(decision_threshold))
    preds = metric_result.pop("preds")
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    row = {
        "split": split_name,
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        "decision_threshold": float(decision_threshold),
        **metric_result,
    }
    add_param_columns(row, params, RNN_PARAM_COLUMNS)

    if not return_predictions:
        return row

    predictions_df = eval_metadata_df.copy()
    predictions_df["score"] = scores
    predictions_df["prediction"] = preds
    predictions_df["decision_threshold"] = float(decision_threshold)
    return row, predictions_df


def evaluate_rnn_config_walk_forward(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    modeled_input_df: pd.DataFrame,
    fold_specs: list[dict],
) -> tuple[dict, list[dict]]:
    fold_rows = []
    for spec in fold_specs:
        fold_row = evaluate_rnn_params(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            train_input_df=subset_by_dates(modeled_input_df, spec["train_dates"]),
            eval_input_df=subset_by_dates(modeled_input_df, spec["validation_dates"]),
            split_name="walk_forward_validation",
            tune_threshold=CONFIG["tune_decision_threshold"],
        )
        fold_rows.append(
            {
                **fold_row,
                "fold": spec["fold"],
                "fold_train_n_dates": spec["train_n_dates"],
                "fold_validation_n_dates": spec["validation_n_dates"],
                "fold_train_date_min": spec["train_date_min"],
                "fold_train_date_max": spec["train_date_max"],
                "fold_validation_date_min": spec["validation_date_min"],
                "fold_validation_date_max": spec["validation_date_max"],
            }
        )

    fold_results_df = pd.DataFrame(fold_rows)
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    primary_metric = CONFIG["primary_validation_metric"]
    metric_mean = float(fold_results_df[primary_metric].mean())
    summary_row = {
        "split": "walk_forward_validation",
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        "balanced_accuracy": metric_mean,
        "accuracy": float(fold_results_df["accuracy"].mean()),
        "f1_score": float(fold_results_df["f1_score"].mean()),
    }
    add_param_columns(summary_row, params, RNN_PARAM_COLUMNS)
    return summary_row, fold_rows



In [8]:
selection_metric = CONFIG["selection_metric"]
walk_forward_grid_rows = []
walk_forward_fold_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    features = FEATURE_SETS[feature_set_name]
    for params in RNN_PARAM_GRID:
        summary_row, fold_rows = evaluate_rnn_config_walk_forward(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            modeled_input_df=modeled_df,
            fold_specs=walk_forward_fold_specs,
        )
        walk_forward_grid_rows.append(summary_row)
        walk_forward_fold_rows.extend(fold_rows)

walk_forward_grid_results_df = pd.DataFrame(walk_forward_grid_rows)
walk_forward_fold_results_df = pd.DataFrame(walk_forward_fold_rows)
if selection_metric not in walk_forward_grid_results_df.columns:
    raise KeyError(f"Selection metric is not available: {selection_metric}")

validation_grid_results_df = walk_forward_grid_results_df.sort_values(
    [selection_metric, "balanced_accuracy", "f1_score", "accuracy", "feature_set", "param_set"],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)


In [9]:
validation_best_by_feature_set_df = ModelReportBuilder.select_best_validation_by_feature_set(
    validation_grid_results_df,
    selection_metric=CONFIG["selection_metric"],
)

validation_best_by_feature_set_report_df = ModelReportBuilder.build_validation_best_by_feature_set_report(
    validation_best_by_feature_set_df,
    param_columns=RNN_PARAM_COLUMNS,
)

validation_best_by_feature_set_report_df


,feature_family,feature_set,n_features,param_set,sequence_length,cell_type,hidden_units,dropout,recurrent_dropout,dense_units,dense_dropout,l2,learning_rate,batch_size,max_epochs,patience,class_weight_mode,validation_accuracy,validation_balanced_accuracy,validation_f1_score
0,price + volume + GDELT sentiment + Reddit + Go...,Model P - price + volume + GDELT sentiment + R...,8,lstm_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_bala...,10,LSTM,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced,0.554122,0.550491,0.507539
1,price + volume,Model B - price + volume | volume percentile r...,5,gru_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced,10,GRU,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced,0.547505,0.550208,0.537736
2,price + volume + GDELT,Model C - price + volume + GDELT | GDELT zscor...,7,gru_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced,10,GRU,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced,0.547057,0.550030,0.607208
3,price + volume + GDELT sentiment + Reddit + Go...,Model P - price + volume + GDELT sentiment + R...,8,gru_len20_h16_drop0p3_l2_1e-3_lr5e-4_b128_bala...,20,GRU,16,0.3,0.0,0,0.0,0.001,0.0005,128,100,10,balanced,0.539641,0.548971,0.580238
4,price + volume + GDELT,Model C - price + volume + GDELT | GDELT senti...,7,gru_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced,10,GRU,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced,0.541140,0.547942,0.559032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,price + volume + Reddit,Model E - price + volume + Reddit | Reddit sen...,7,gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b1...,10,GRU,16,0.3,0.0,8,0.2,0.010,0.0005,128,100,10,balanced,0.520985,0.538371,0.479098
57,price + volume,Model B - price + volume | volume log1p zscore...,5,gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b1...,10,GRU,16,0.3,0.0,8,0.2,0.010,0.0005,128,100,10,balanced,0.536296,0.538277,0.400550
58,price + volume,Model B - price + volume | volume zscore 20d,5,gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b1...,10,GRU,16,0.3,0.0,8,0.2,0.010,0.0005,128,100,10,balanced,0.512305,0.537650,0.465640
59,price + volume + all alternative data,Model F - price + volume + all alternative dat...,10,gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b1...,10,GRU,16,0.3,0.0,8,0.2,0.010,0.0005,128,100,10,balanced,0.538567,0.537300,0.391823


In [10]:
best_validation_params_df = validation_best_by_feature_set_df.copy()


In [11]:
threshold_calibration_rows = []
test_rows = []

for row in best_validation_params_df.to_dict(orient="records"):
    params = params_from_result_row(row)
    feature_set_name = row["feature_set"]
    calibration_row = evaluate_rnn_params(
        feature_set_name=feature_set_name,
        features=FEATURE_SETS[feature_set_name],
        params=params,
        train_input_df=train_df,
        eval_input_df=validation_df,
        split_name="validation_threshold_calibration",
        tune_threshold=True,
    )
    threshold_calibration_rows.append(calibration_row)
    test_rows.append(
        evaluate_rnn_params(
            feature_set_name=feature_set_name,
            features=FEATURE_SETS[feature_set_name],
            params=params,
            train_input_df=train_validation_df,
            eval_input_df=test_df,
            split_name="test_refit_train_validation_selected_threshold",
            decision_threshold=calibration_row["decision_threshold"],
            tune_threshold=False,
        )
    )

threshold_calibration_results_df = pd.DataFrame(threshold_calibration_rows)
test_best_validation_params_df = pd.DataFrame(test_rows).sort_values(
    ["balanced_accuracy", "f1_score", "accuracy", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

(
    simple_hyperparameter_summary_df,
    baseline_walk_forward_row,
    baseline_calibration_row,
    baseline_test_row,
) = ModelReportBuilder.build_simple_hyperparameter_summary(
    best_validation_params_df=best_validation_params_df,
    threshold_calibration_results_df=threshold_calibration_results_df,
    test_best_validation_params_df=test_best_validation_params_df,
    baseline_feature_set=BASELINE_FEATURE_SET,
)


In [12]:
validation_selected_family_test_report_df = ModelReportBuilder.build_validation_selected_family_test_report(
    simple_hyperparameter_summary_df,
    param_columns=RNN_PARAM_COLUMNS,
)

validation_selected_family_test_report_df


,feature_family,feature_set,n_features,param_set,sequence_length,cell_type,hidden_units,dropout,recurrent_dropout,dense_units,dense_dropout,l2,learning_rate,batch_size,max_epochs,patience,class_weight_mode,validation_accuracy,validation_balanced_accuracy,validation_f1_score,test_accuracy,test_balanced_accuracy,test_f1_score,price_volume_baseline_feature_set,price_volume_baseline_test_balanced_accuracy,test_balanced_accuracy_change_vs_price_volume
0,price + volume,Model B - price + volume | volume percentile r...,5,gru_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_balanced,10,GRU,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced,0.547505,0.550208,0.537736,0.517497,0.530177,0.388441,Model B - price + volume | volume percentile r...,0.530177,0.000000
1,price only,Model A - price only,4,gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b1...,10,GRU,16,0.3,0.0,8,0.2,0.010,0.0005,128,100,10,balanced,0.548569,0.542055,0.459504,0.507953,0.521396,0.364384,Model B - price + volume | volume percentile r...,0.530177,-0.008782
2,price + volume + Google,Model G - price + volume + Google | Google zsc...,6,gru_len20_h16_drop0p3_l2_1e-3_lr5e-4_b128_bala...,20,GRU,16,0.3,0.0,0,0.0,0.001,0.0005,128,100,10,balanced,0.558080,0.546448,0.476750,0.519088,0.520241,0.521877,Model B - price + volume | volume percentile r...,0.530177,-0.009936
3,price + volume + Google score attention,Model L - price + volume + Google score attent...,6,gru_len20_h16_drop0p3_l2_1e-3_lr5e-4_b128_bala...,20,GRU,16,0.3,0.0,0,0.0,0.001,0.0005,128,100,10,balanced,0.558080,0.546448,0.476750,0.519088,0.520241,0.521877,Model B - price + volume | volume percentile r...,0.530177,-0.009936
4,price + volume + GDELT attention,Model J - price + volume + GDELT attention | G...,6,gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b1...,10,GRU,16,0.3,0.0,8,0.2,0.010,0.0005,128,100,10,balanced,0.540278,0.542653,0.384265,0.523860,0.519416,0.571565,Model B - price + volume | volume percentile r...,0.530177,-0.010761
5,price + volume + Reddit attention,Model K - price + volume + Reddit attention | ...,6,gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b1...,10,GRU,16,0.3,0.0,8,0.2,0.010,0.0005,128,100,10,balanced,0.542680,0.544588,0.397465,0.520679,0.517636,0.558162,Model B - price + volume | volume percentile r...,0.530177,-0.012541
6,price + volume + Reddit,Model E - price + volume + Reddit | Reddit per...,7,lstm_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_bala...,10,LSTM,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced,0.541260,0.546671,0.449651,0.528632,0.513460,0.641098,Model B - price + volume | volume percentile r...,0.530177,-0.016717
7,price + volume + all alternative data,Model F - price + volume + all alternative dat...,10,gru_len10_h16_dense8_drop0p3_l2_1e-2_lr5e-4_b1...,10,GRU,16,0.3,0.0,8,0.2,0.010,0.0005,128,100,10,balanced,0.538567,0.537300,0.391823,0.520148,0.513185,0.585812,Model B - price + volume | volume percentile r...,0.530177,-0.016992
8,price + volume + GDELT sentiment + Reddit atte...,Model O - price + volume + GDELT sentiment + R...,7,lstm_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_bala...,10,LSTM,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced,0.534455,0.545824,0.419633,0.522800,0.511756,0.613734,Model B - price + volume | volume percentile r...,0.530177,-0.018421
9,price + volume + all attention,Model N - price + volume + all attention | zsc...,8,lstm_len10_h8_drop0p2_l2_1e-3_lr1e-3_b128_bala...,10,LSTM,8,0.2,0.0,0,0.0,0.001,0.0010,128,80,8,balanced,0.523290,0.543296,0.554627,0.499470,0.505614,0.450524,Model B - price + volume | volume percentile r...,0.530177,-0.024563
